# TrialsIntel Demo
End-to-end walkthrough: fetch → extract → store → search

In [ ]:
import sys; sys.path.insert(0, '..')
from src.fetch_trials import fetch_trials_from_api

trials = fetch_trials_from_api(query='cancer', max_results=3)
print(f'Fetched {len(trials)} trials')
trials[0]

In [ ]:
from src.extract_entities import extract_with_claude

trial = trials[0]
text = f"{trial['title']}. {trial['summary']}"
entities = extract_with_claude(text)
entities

In [ ]:
import sqlite3
from src.database import init_db, save_trial, get_all_trials

conn = sqlite3.connect(':memory:')
init_db(conn=conn)
for t in trials:
    save_trial({**t, **extract_with_claude(f"{t['title']}. {t['summary']}")}, conn=conn)

df = get_all_trials(conn=conn)
df[['nct_id', 'title', 'drug', 'disease', 'phase']]

In [ ]:
from src.search import search_by_keywords

results = search_by_keywords(df, disease='cancer')
print(f'Cancer trials: {len(results)}')
results[['nct_id', 'drug', 'disease', 'phase']]